In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,141,34.418803
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,141,22.478222
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,141,39.344682
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,141,47.668795
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,141,38.440565
...,...,...,...,...,...,...,...,...,...,...,...
1079995,person_time,impairment,anemia,severe,95_plus,severe,1,zero,0,129,0.000000
1079996,person_time,impairment,anemia,severe,95_plus,severe,2,zero,0,129,0.000000
1079997,person_time,impairment,anemia,severe,95_plus,severe,3,zero,0,129,0.000000
1079998,person_time,impairment,anemia,severe,95_plus,severe,4,zero,0,129,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    270000
mild          270000
moderate      270000
severe        270000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.557211e+06
              2                  2.656243e+06
              3                  2.236837e+06
              4                  1.820072e+06
              5                  1.593141e+06
intervention  1                  2.557257e+06
              2                  2.656292e+06
              3                  2.236874e+06
              4                  1.820104e+06
              5                  1.593162e+06
zero          1                  2.557211e+06
              2                  2.656243e+06
              3                  2.236837e+06
              4                  1.820072e+06
              5                  1.593141e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  1.388929e+06
              2                  1.487662e+06
              3                  1.138368e+06
              4                  8.176504e+05
              5                  6.704840e+05
intervention  1                  1.193066e+06
              2                  1.292610e+06
              3                  9.715829e+05
              4                  6.949018e+05
              5                  5.558249e+05
zero          1                  1.388929e+06
              2                  1.487662e+06
              3                  1.138368e+06
              4                  8.176504e+05
              5                  6.704840e+05
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.543142
              2                  0.560063
              3                  0.508919
              4                  0.449241
              5                  0.420857
intervention  1                  0.466541
              2                  0.486622
              3                  0.434348
              4                  0.381792
              5                  0.348881
zero          1                  0.543142
              2                  0.560063
              3                  0.508919
              4                  0.449241
              5                  0.420857
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,17452.628925
1,Female,0.0,0.019178,not_pregnant,2,17380.997496
2,Female,0.0,0.019178,not_pregnant,3,16409.683914
3,Female,0.0,0.019178,not_pregnant,4,14424.706826
4,Female,0.0,0.019178,not_pregnant,5,13407.888483
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,5056.112334
281,Male,95.0,125.000000,not_pregnant,2,4424.735951
282,Male,95.0,125.000000,not_pregnant,3,4548.244848
283,Male,95.0,125.000000,not_pregnant,4,4870.219344


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    2.372743e+06
2    2.463522e+06
3    2.073741e+06
4    1.684268e+06
5    1.470164e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.288737e+06
              2                  1.379726e+06
              3                  1.055366e+06
              4                  7.566418e+05
              5                  6.187286e+05
intervention  1                  1.106983e+06
              2                  1.198804e+06
              3                  9.007261e+05
              4                  6.430407e+05
              5                  5.129131e+05
zero          1                  1.288737e+06
              2                  1.379726e+06
              3                  1.055366e+06
              4                  7.566418e+05
              5                  6.187286e+05
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,141,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,141,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,141,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,141,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,zero,0,129,0.0
539996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,zero,0,129,0.0
539997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,zero,0,129,0.0
539998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,zero,0,129,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.838842e+06
              2                  2.728059e+06
              3                  1.833772e+06
              4                  1.639386e+06
              5                  1.102531e+06
intervention  1                  2.726292e+06
              2                  2.607677e+06
              3                  1.743421e+06
              4                  1.569062e+06
              5                  1.049705e+06
zero          1                  2.838842e+06
              2                  2.728059e+06
              3                  1.833772e+06
              4                  1.639386e+06
              5                  1.102531e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,224.733516
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,223.105013
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,188.906434
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,141.679826
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,117.252269
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,154.707855
23996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,175.878404
23997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,143.308329
23998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,113.995262


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  191414.329768
              2                  194285.381864
              3                  161615.968307
              4                  129654.953878
              5                  113172.867511
intervention  1                  190419.313982
              2                  193357.134732
              3                  161085.076088
              4                  129327.624626
              5                  112881.365341
zero          1                  191414.329768
              2                  194285.381864
              3                  161615.968307
              4                  129654.953878
              5                  113172.867511
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,14769.329314,zero
1,Female,0.0,0.019178,2,14182.900679,zero
2,Female,0.0,0.019178,3,12601.737191,zero
3,Female,0.0,0.019178,4,10966.456900,zero
4,Female,0.0,0.019178,5,9094.030016,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,3805.842571,intervention
746,Male,95.0,125.000000,2,3269.974294,intervention
747,Male,95.0,125.000000,3,3322.922486,intervention
748,Male,95.0,125.000000,4,3557.999580,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.614290e+07
              2                  2.354700e+07
              3                  2.153242e+07
              4                  2.088089e+07
              5                  1.685034e+07
intervention  1                  2.345295e+07
              2                  2.074621e+07
              3                  1.874619e+07
              4                  1.805516e+07
              5                  1.425888e+07
zero          1                  2.614290e+07
              2                  2.354700e+07
              3                  2.153242e+07
              4                  2.088089e+07
              5                  1.685034e+07
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.743164e+07
              2                  2.492673e+07
              3                  2.258779e+07
              4                  2.163753e+07
              5                  1.746907e+07
intervention  1                  2.455993e+07
              2                  2.194502e+07
              3                  1.964692e+07
              4                  1.869820e+07
              5                  1.477179e+07
zero          1                  2.743164e+07
              2                  2.492673e+07
              3                  2.258779e+07
              4                  2.163753e+07
              5                  1.746907e+07
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  7596.117757
              2                  7767.503194
              3                  7031.727783
              4                  6232.914051
              5                  5474.484244
baseline      1                  7596.117757
              2                  7767.503194
              3                  7031.727783
              4                  6232.914051
              5                  5474.484244
intervention  1                  2666.666586
              2                  2903.385295
              3                  3222.445471
              4                  3225.543108
              5                  2975.408609
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)